In [ ]:
pip install pandas numpy scikit-learn textblob transformers torch

In [ ]:
"""
Deceptive Review Detection System
===================================
Detects computer-generated (CG) vs original (OR) reviews using:
  - Stylometric / linguistic features
  - Sentiment polarity features
  - TF-IDF representations
  - Logistic Regression baseline
  - BERT-based transformer classifier

Label convention:  OR = original/genuine,  CG = computer-generated/deceptive
Binary target:     1 = CG (deceptive),     0 = OR (genuine)

Requirements:
    pip install pandas numpy scikit-learn textblob transformers torch
    (GPU strongly recommended for BERT training)
"""

# ─────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────────────────────
import re
import warnings
import numpy as np
import pandas as pd

from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils import shuffle

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW                          # moved here in transformers ≥4.x
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Download TextBlob / NLTK corpora if not already present
import nltk
nltk.download("punkt",                          quiet=True)
nltk.download("punkt_tab",                      quiet=True)
nltk.download("averaged_perceptron_tagger",     quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("brown",                          quiet=True)
nltk.download("wordnet",                        quiet=True)
nltk.download("omw-1.4",                        quiet=True)

# ─────────────────────────────────────────────────────────────
# 1.  DATA LOADING & PREPROCESSING
# ─────────────────────────────────────────────────────────────
DATA_PATH = "fake reviews dataset.csv"   # ← update path if needed
RANDOM_STATE = 42
TEST_SIZE = 0.20


def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.rename(columns={"text_": "text"}, inplace=True)
    df.dropna(subset=["text", "label"], inplace=True)
    df["text"] = df["text"].astype(str).str.strip()
    # Binary target: CG = 1 (deceptive), OR = 0 (genuine)
    df["target"] = (df["label"] == "CG").astype(int)
    return df


# ─────────────────────────────────────────────────────────────
# 2.  FEATURE ENGINEERING  (stylometry + sentiment)
# ─────────────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)          # remove URLs
    text = re.sub(r"[^a-z0-9\s.,!?']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_stylometric_features(text: str) -> dict:
    """
    Stylometry + linguistic cues that distinguish human vs machine text.
    Research shows CG text tends to be more verbose, uniform in sentence
    length, and overuse superlatives / exclamations.
    """
    blob = TextBlob(text)
    sentences = blob.sentences
    words = blob.words

    num_words = len(words) if words else 1
    num_sentences = len(sentences) if sentences else 1
    num_chars = len(text)

    # --- sentence-level ---
    sent_lengths = [len(str(s).split()) for s in sentences] if sentences else [0]
    avg_sent_len = np.mean(sent_lengths)
    std_sent_len = np.std(sent_lengths)      # low std → suspiciously uniform

    # --- lexical richness ---
    unique_words = len(set(w.lower() for w in words))
    type_token_ratio = unique_words / num_words

    # --- punctuation / exclamation abuse (common in CG) ---
    exclamation_count = text.count("!")
    question_count = text.count("?")
    exclamation_ratio = exclamation_count / num_sentences

    # --- personal pronoun usage (humans use "I" more naturally) ---
    first_person = sum(1 for w in words if w.lower() in {"i", "my", "me", "mine", "myself"})
    first_person_ratio = first_person / num_words

    # --- sentiment polarity & subjectivity ---
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    # --- average word length (CG text can differ) ---
    avg_word_len = np.mean([len(w) for w in words]) if words else 0

    # --- capital letter ratio (CG sometimes shouts) ---
    capital_ratio = sum(1 for c in text if c.isupper()) / max(num_chars, 1)

    # --- repetition: ratio of word frequencies (detect copy-paste style) ---
    from collections import Counter
    word_freq = Counter(w.lower() for w in words)
    repetition_ratio = (
        sum(v for v in word_freq.values() if v > 1) / num_words
        if num_words > 1 else 0
    )

    return {
        "num_words": num_words,
        "num_sentences": num_sentences,
        "num_chars": num_chars,
        "avg_sent_len": avg_sent_len,
        "std_sent_len": std_sent_len,
        "type_token_ratio": type_token_ratio,
        "exclamation_ratio": exclamation_ratio,
        "question_count": question_count,
        "first_person_ratio": first_person_ratio,
        "polarity": polarity,
        "subjectivity": subjectivity,
        "avg_word_len": avg_word_len,
        "capital_ratio": capital_ratio,
        "repetition_ratio": repetition_ratio,
    }


def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    print("⚙  Extracting stylometric features …")
    feats = df["text"].apply(lambda t: extract_stylometric_features(t))
    return pd.DataFrame(feats.tolist())


# ─────────────────────────────────────────────────────────────
# 3.  LOGISTIC REGRESSION PIPELINE (TF-IDF + stylometric)
# ─────────────────────────────────────────────────────────────

def run_logistic_regression(df: pd.DataFrame):
    print("\n" + "="*60)
    print("  MODEL 1: LOGISTIC REGRESSION (TF-IDF + Stylometric)")
    print("="*60)

    df = shuffle(df, random_state=RANDOM_STATE).reset_index(drop=True)
    X_text = df["text"].apply(clean_text)
    X_style = build_feature_matrix(df)
    y = df["target"]

    # Split
    idx_train, idx_test = train_test_split(
        df.index, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    # --- TF-IDF component ---
    tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                             sublinear_tf=True, min_df=3)
    X_tfidf_train = tfidf.fit_transform(X_text.iloc[idx_train])
    X_tfidf_test  = tfidf.transform(X_text.iloc[idx_test])

    # --- Stylometric component ---
    scaler = StandardScaler()
    X_style_train = scaler.fit_transform(X_style.iloc[idx_train])
    X_style_test  = scaler.transform(X_style.iloc[idx_test])

    from scipy.sparse import hstack, csr_matrix
    X_train = hstack([X_tfidf_train, csr_matrix(X_style_train)])
    X_test  = hstack([X_tfidf_test,  csr_matrix(X_style_test)])

    y_train = y.iloc[idx_train]
    y_test  = y.iloc[idx_test]

    # Train
    clf = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced",
                              solver="lbfgs", random_state=RANDOM_STATE)
    clf.fit(X_train, y_train)

    y_pred  = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["OR (genuine)", "CG (deceptive)"]))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=["OR (genuine)", "CG (deceptive)"])
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title("Logistic Regression — Confusion Matrix")
    plt.tight_layout()
    plt.savefig("lr_confusion_matrix.png", dpi=150)
    plt.close()
    print("📊  Confusion matrix saved → lr_confusion_matrix.png")

    # Top deceptive TF-IDF features
    feature_names = tfidf.get_feature_names_out()
    n_style = X_style_train.shape[1]
    coef = clf.coef_[0]
    tfidf_coef = coef[:len(feature_names)]
    top_idx = np.argsort(tfidf_coef)[::-1][:15]
    print("\nTop 15 TF-IDF features pushing toward CG (deceptive):")
    for i in top_idx:
        print(f"   '{feature_names[i]}': {tfidf_coef[i]:+.4f}")

    return clf, tfidf, scaler


# ─────────────────────────────────────────────────────────────
# 4.  BERT FINE-TUNING
# ─────────────────────────────────────────────────────────────

BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LEN    = 128
BERT_BATCH_SIZE = 32
BERT_EPOCHS     = 3
BERT_LR         = 2e-5


class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }


def run_bert(df: pd.DataFrame):
    print("\n" + "="*60)
    print("  MODEL 2: BERT FINE-TUNING")
    print("="*60)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Using device: {device}")
    if device.type == "cpu":
        print("  ⚠  No GPU detected — BERT training will be slow.")
        print("     Consider running on Colab/Kaggle with a GPU runtime.")

    df = shuffle(df, random_state=RANDOM_STATE).reset_index(drop=True)
    texts  = df["text"].tolist()
    labels = df["target"].tolist()

    train_texts, test_texts, train_labels, test_labels = train_test_split(
        texts, labels, test_size=TEST_SIZE,
        random_state=RANDOM_STATE, stratify=labels
    )

    tokenizer = BertTokenizerFast.from_pretrained(BERT_MODEL_NAME)

    train_ds = ReviewDataset(train_texts, train_labels, tokenizer, BERT_MAX_LEN)
    test_ds  = ReviewDataset(test_texts,  test_labels,  tokenizer, BERT_MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=BERT_BATCH_SIZE, shuffle=True,  num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=BERT_BATCH_SIZE, shuffle=False, num_workers=2)

    model = BertForSequenceClassification.from_pretrained(
        BERT_MODEL_NAME, num_labels=2
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=BERT_LR, eps=1e-8)
    total_steps = len(train_loader) * BERT_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    # ── Training loop ──
    train_losses, val_aucs = [], []

    for epoch in range(1, BERT_EPOCHS + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_b       = batch["label"].to(device)

            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels_b)
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        avg_loss = total_loss / len(train_loader)
        train_losses.append(avg_loss)

        # Eval
        model.eval()
        all_preds, all_probs, all_labels_e = [], [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                logits = model(input_ids=input_ids,
                               attention_mask=attention_mask).logits
                probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                preds  = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_probs.extend(probs)
                all_labels_e.extend(batch["label"].numpy())

        auc = roc_auc_score(all_labels_e, all_probs)
        val_aucs.append(auc)
        print(f"  Epoch {epoch}/{BERT_EPOCHS}  —  Loss: {avg_loss:.4f}  |  Val ROC-AUC: {auc:.4f}")

    print("\nFinal Classification Report (BERT):")
    print(classification_report(all_labels_e, all_preds,
                                  target_names=["OR (genuine)", "CG (deceptive)"]))
    print(f"Final ROC-AUC: {val_aucs[-1]:.4f}")

    # Save model
    model.save_pretrained("bert_deception_model")
    tokenizer.save_pretrained("bert_deception_model")
    print("💾  BERT model saved → bert_deception_model/")

    # Training curve
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(range(1, BERT_EPOCHS+1), train_losses, marker="o", color="tomato")
    ax1.set_title("BERT Training Loss")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax2.plot(range(1, BERT_EPOCHS+1), val_aucs, marker="o", color="steelblue")
    ax2.set_title("BERT Validation ROC-AUC")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("AUC")
    plt.tight_layout()
    plt.savefig("bert_training_curve.png", dpi=150)
    plt.close()
    print("📊  Training curves saved → bert_training_curve.png")

    return model, tokenizer


# ─────────────────────────────────────────────────────────────
# 5.  DECEPTION SCORE INFERENCE  (use either model)
# ─────────────────────────────────────────────────────────────

def predict_deception_lr(
    text: str,
    clf, tfidf, scaler,
    threshold: float = 0.5
) -> dict:
    """
    Returns a deception score and key indicators for a single review
    using the Logistic Regression model.
    """
    from scipy.sparse import hstack, csr_matrix

    clean  = clean_text(text)
    tfidf_vec = tfidf.transform([clean])
    style_dict = extract_stylometric_features(text)
    style_vec  = scaler.transform(pd.DataFrame([style_dict]))

    X = hstack([tfidf_vec, csr_matrix(style_vec)])
    proba = clf.predict_proba(X)[0, 1]
    label = "CG (DECEPTIVE)" if proba >= threshold else "OR (GENUINE)"

    # Key stylometric indicators
    indicators = {
        "sentiment_polarity":    round(style_dict["polarity"], 3),
        "subjectivity":          round(style_dict["subjectivity"], 3),
        "exclamation_ratio":     round(style_dict["exclamation_ratio"], 3),
        "type_token_ratio":      round(style_dict["type_token_ratio"], 3),
        "first_person_ratio":    round(style_dict["first_person_ratio"], 3),
        "std_sentence_length":   round(style_dict["std_sent_len"], 3),
        "repetition_ratio":      round(style_dict["repetition_ratio"], 3),
    }

    return {
        "deception_score": round(float(proba), 4),
        "prediction":      label,
        "key_indicators":  indicators,
    }


def predict_deception_bert(
    text: str,
    model, tokenizer,
    device=None,
    threshold: float = 0.5
) -> dict:
    """
    Returns a deception score and key indicators for a single review
    using the fine-tuned BERT model.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    enc = tokenizer(
        text, max_length=BERT_MAX_LEN,
        padding="max_length", truncation=True,
        return_tensors="pt"
    )
    input_ids      = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    model.eval()
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        proba  = torch.softmax(logits, dim=1)[0, 1].item()

    label = "CG (DECEPTIVE)" if proba >= threshold else "OR (GENUINE)"

    # Supplement with stylometric indicators
    style_dict = extract_stylometric_features(text)
    indicators = {
        "sentiment_polarity":  round(style_dict["polarity"], 3),
        "subjectivity":        round(style_dict["subjectivity"], 3),
        "exclamation_ratio":   round(style_dict["exclamation_ratio"], 3),
        "type_token_ratio":    round(style_dict["type_token_ratio"], 3),
        "first_person_ratio":  round(style_dict["first_person_ratio"], 3),
        "std_sentence_length": round(style_dict["std_sent_len"], 3),
        "repetition_ratio":    round(style_dict["repetition_ratio"], 3),
    }

    return {
        "deception_score": round(proba, 4),
        "prediction":      label,
        "key_indicators":  indicators,
    }


# ─────────────────────────────────────────────────────────────
# 6.  MAIN
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # ── Load data ──
    print("📂  Loading dataset …")
    df = load_data(DATA_PATH)
    print(f"    Loaded {len(df):,} rows  |  CG: {df.target.sum():,}  OR: {(df.target==0).sum():,}")

    # ── Model 1: Logistic Regression ──
    clf, tfidf, scaler = run_logistic_regression(df)

    # ── Quick inference demo with LR ──
    sample_reviews = [
        "This product changed my life! Absolutely amazing!! Best purchase ever!!!",   # likely CG
        "The seams started coming apart after two washes. Pretty disappointed.",        # likely OR
    ]
    print("\n── Deception Score Demo (Logistic Regression) ──")
    for rev in sample_reviews:
        result = predict_deception_lr(rev, clf, tfidf, scaler)
        print(f"\n  Review : {rev[:80]}…")
        print(f"  Score  : {result['deception_score']}  →  {result['prediction']}")
        print(f"  Indicators: {result['key_indicators']}")

    # ── Model 2: BERT (comment out if no GPU / time constrained) ──
    RUN_BERT = True   # ← set False to skip BERT training
    if RUN_BERT:
        bert_model, bert_tokenizer = run_bert(df)

        print("\n── Deception Score Demo (BERT) ──")
        for rev in sample_reviews:
            result = predict_deception_bert(rev, bert_model, bert_tokenizer)
            print(f"\n  Review : {rev[:80]}…")
            print(f"  Score  : {result['deception_score']}  →  {result['prediction']}")
            print(f"  Indicators: {result['key_indicators']}")

    print("\n✅  Done.")

📂  Loading dataset …
    Loaded 40,432 rows  |  CG: 20,216  OR: 20,216

  MODEL 1: LOGISTIC REGRESSION (TF-IDF + Stylometric)
⚙  Extracting stylometric features …

Classification Report:
                precision    recall  f1-score   support

  OR (genuine)       0.93      0.94      0.94      4044
CG (deceptive)       0.94      0.93      0.94      4043

      accuracy                           0.94      8087
     macro avg       0.94      0.94      0.94      8087
  weighted avg       0.94      0.94      0.94      8087

ROC-AUC: 0.9845
📊  Confusion matrix saved → lr_confusion_matrix.png

Top 15 TF-IDF features pushing toward CG (deceptive):
   'the only': +6.6072
   'will keep': +5.6658
   'and the': +4.7779
   'and it': +4.2305
   'that it': +4.2146
   'has the': +3.9965
   'very good': +3.8590
   'but it': +3.7867
   'an': +3.6828
   'couple': +3.4741
   'only reason': +3.3263
   'wide': +3.2740
   'comfortable': +3.2195
   'it been': +3.2028
   'couple of': +3.0956

── Deception Sco

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/3  —  Loss: 0.1792  |  Val ROC-AUC: 0.9952
  Epoch 2/3  —  Loss: 0.0436  |  Val ROC-AUC: 0.9963
  Epoch 3/3  —  Loss: 0.0146  |  Val ROC-AUC: 0.9973

Final Classification Report (BERT):
                precision    recall  f1-score   support

  OR (genuine)       0.99      0.93      0.96      4044
CG (deceptive)       0.93      0.99      0.96      4043

      accuracy                           0.96      8087
     macro avg       0.96      0.96      0.96      8087
  weighted avg       0.96      0.96      0.96      8087

Final ROC-AUC: 0.9973


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾  BERT model saved → bert_deception_model/
📊  Training curves saved → bert_training_curve.png

── Deception Score Demo (BERT) ──

  Review : This product changed my life! Absolutely amazing!! Best purchase ever!!!…
  Score  : 0.0013  →  OR (GENUINE)
  Indicators: {'sentiment_polarity': 0.969, 'subjectivity': 0.6, 'exclamation_ratio': 1.5, 'type_token_ratio': 1.0, 'first_person_ratio': 0.1, 'std_sentence_length': np.float64(1.479), 'repetition_ratio': 0.0}

  Review : The seams started coming apart after two washes. Pretty disappointed.…
  Score  : 0.0002  →  OR (GENUINE)
  Indicators: {'sentiment_polarity': -0.25, 'subjectivity': 0.875, 'exclamation_ratio': 0.0, 'type_token_ratio': 1.0, 'first_person_ratio': 0.0, 'std_sentence_length': np.float64(3.0), 'repetition_ratio': 0.0}

✅  Done.
